# Dask Exercise 2: dask delayed

* [dask delayed](https://docs.dask.org/en/stable/delayed.html)
* [dask tutorial](https://tutorial.dask.org/03_dask.delayed.html)

Skills:
* Convert a for loop into a simple dask delayed workflow 
* Get more familiar with `dask.dataframe` wrangling

In [1]:
import dask.dataframe as dd
import pandas as pd

from dask import delayed, compute

GCS_FILE_PATH = ("gs://calitp-analytics-data/data-analyses/"
                 "rt_delay/v2_rt_trips/"
                )

analysis_date = "2023-03-15"
la_metro = 182
big_blue_bus = 300
muni = 282

operators = [la_metro, big_blue_bus, muni]

## Simple Workflow to Parallelize

This is a typical workflow. 
1. Read in pandas df.
2. Apply a certain function.
3. Export df.

Let's say we have a df corresponding to each operator. We want to apply the same aggregation function and then save out the results.

Typically, we would use a loop. A loop is **sequential**. By using `dask delayed` objects, we can run those **simultaneously**. Instead of running operator 1, operator 2, operator 3, ... , operator N, why not let them run at the same time and save out the results?

There is nothing inherent in our workflow that specifies that operator 1 must be run before operator 2. We are applying the same function to each operator. To speed it up, let's use dask to run it in parallel and get our results.

In [2]:
df = pd.read_parquet(
    f"{GCS_FILE_PATH}{big_blue_bus}_{analysis_date}.parquet")

In [43]:
df.head()

,feed_key,trip_key,gtfs_dataset_key,activity_date,trip_id,route_id,route_short_name,shape_id,direction_id,route_type,route_long_name,route_desc,calitp_itp_id,median_time,direction,mean_speed_mph,organization_name
0,4f77ef02b983eccc0869c7540f98a7d0,f088a730653c0679da62feaa9c25bc26,dbbe8ee4864a2715a40749605395d584,2023-03-15,893061,3554,1,26158,0,3,Main St & Santa Monica Blvd/UCLA,None,300,10:16:03.500000,Northbound,8.925867,City of Santa Monica
1,4f77ef02b983eccc0869c7540f98a7d0,8451ebd43e9159cc27b03ae2ae868ee3,dbbe8ee4864a2715a40749605395d584,2023-03-15,894836,3564,R12,26195,0,3,Venice/Westwood Sta/UCLA Rapid,None,300,14:30:33,Northbound,12.339312,City of Santa Monica
2,4f77ef02b983eccc0869c7540f98a7d0,c3ff3523997c3c43495ebdad117f95a0,dbbe8ee4864a2715a40749605395d584,2023-03-15,893098,3554,1,26158,0,3,Main St & Santa Monica Blvd/UCLA,None,300,16:26:50,Northbound,8.208192,City of Santa Monica
3,4f77ef02b983eccc0869c7540f98a7d0,b4d78606e25155aa02b25a26e008374e,dbbe8ee4864a2715a40749605395d584,2023-03-15,893102,3554,1,26158,0,3,Main St & Santa Monica Blvd/UCLA,None,300,17:06:58,Northbound,7.702262,City of Santa Monica
4,4f77ef02b983eccc0869c7540f98a7d0,12276a7ee0019ab61b04929e64f0b444,dbbe8ee4864a2715a40749605395d584,2023-03-15,893173,3554,1,26164,1,3,Main St & Santa Monica Blvd/UCLA,None,300,13:00:54,Southbound,6.995567,City of Santa Monica


In [3]:
# Set up a function that counts the number of 
# unique route_ids and route_type
def simple_route_aggregation(df: pd.DataFrame) -> pd.DataFrame:
        aggregated = (df.groupby(["calitp_itp_id",
                                  "organization_name"])
                      .agg({"route_id": "nunique", 
                            "route_type": "nunique"})
                      .reset_index()
                     )
        
        return aggregated


In [4]:
df_agg = simple_route_aggregation(df)

In [5]:
df_agg

,calitp_itp_id,organization_name,route_id,route_type
0,300,City of Santa Monica,18,1


### Move it to delayed

We can use the `@delayed` decorator right above our defined function.


Alternatively, you can wrap the function, like `delayed(my_function)(args)`. These are equivalent.

```
@delayed
def my_function(df):
    df2 = do something
    return df2
    
    
or...
delayed(my_function)(df)
```

**Note where the parentheses fall**...it is not a typo.


In [6]:
# We can use a decorator to make it a delayed function
@delayed
def import_data(itp_id: int):
    return pd.read_parquet(
        f"{GCS_FILE_PATH}{itp_id}_{analysis_date}.parquet")

In [7]:
# We have a list of 3 operators we would have looped over
operators

[182, 300, 282]

In [8]:
# Let's read in our data using list comprehension
dfs = [import_data(x) for x in operators]

In [9]:
# We have a list of delayed objects
# these dfs are not materialized / read into memory
dfs

[Delayed('import_data-5bc02365-7c9d-4f88-928e-b35a56c93c51'),
 Delayed('import_data-47246a7a-5010-48b9-a87d-b0083c565099'),
 Delayed('import_data-8af6af1c-d62e-4a7a-aaf1-c58f8983c1c2')]

In [10]:
# Set  up a list to store our results
results = [delayed(simple_route_aggregation)(df) for df in dfs]

In [11]:
# The results are also delayed objects
results

[Delayed('simple_route_aggregation-3c6349e0-57bd-41d8-b2fc-910aee5ae08d'),
 Delayed('simple_route_aggregation-4f1ed816-b133-4b10-98c9-68cbcd9a6799'),
 Delayed('simple_route_aggregation-afe0a201-f74e-4923-8b45-1646df2620c0')]

In [12]:
# Wrap compute around each of the items in the results list 
# and see what's inside
results_computed = [compute(i) for i in results]

In [13]:
# This is a list of tuples...that's not what we want
results_computed

[(   calitp_itp_id                                  organization_name  route_id  \
  0            182  Los Angeles County Metropolitan Transportation...       120   
  
     route_type  
  0           3  ,),
 (   calitp_itp_id     organization_name  route_id  route_type
  0            300  City of Santa Monica        18           1,),
 (   calitp_itp_id                 organization_name  route_id  route_type
  0            282  City and County of San Francisco        67           3,)]

In [14]:
type(results_computed[0])

tuple

In [15]:
# We need the first item of the tuple...that's our df
type(results_computed[0][0])

pandas.core.frame.DataFrame

In [16]:
results_computed_correct = [compute(i)[0] for i in results]

In [17]:
results_computed_correct

[   calitp_itp_id                                  organization_name  route_id  \
 0            182  Los Angeles County Metropolitan Transportation...       120   
 
    route_type  
 0           3  ,
    calitp_itp_id     organization_name  route_id  route_type
 0            300  City of Santa Monica        18           1,
    calitp_itp_id                 organization_name  route_id  route_type
 0            282  City and County of San Francisco        67           3]

In [18]:
type(results_computed_correct[0])

pandas.core.frame.DataFrame

In [19]:
results_computed_correct[0]

,calitp_itp_id,organization_name,route_id,route_type
0,182,Los Angeles County Metropolitan Transportation...,120,3


In [44]:
results_computed_correct[1]

,calitp_itp_id,organization_name,route_id,route_type
0,300,City of Santa Monica,18,1


In [20]:
# Alternatively, the code can be written like a loop, 
# but it won't run like a loop. It will run it simultaneously 
# for the three operators

results2 = []

for itp_id in operators:
    operator_df = import_data(itp_id)
    print(f"type for operator_df: {type(operator_df)}")
    
    aggregated_df = delayed(simple_route_aggregation)(operator_df)
    print(f"type for aggregated_df: {type(aggregated_df)}")
    
    results2.append(aggregated_df)


type for operator_df: <class 'dask.delayed.Delayed'>
type for aggregated_df: <class 'dask.delayed.Delayed'>
type for operator_df: <class 'dask.delayed.Delayed'>
type for aggregated_df: <class 'dask.delayed.Delayed'>
type for operator_df: <class 'dask.delayed.Delayed'>
type for aggregated_df: <class 'dask.delayed.Delayed'>


In [21]:
results_computed2 = [compute(i)[0] for i in results2]

In [22]:
results_computed2[2]

,calitp_itp_id,organization_name,route_id,route_type
0,282,City and County of San Francisco,67,3


In [23]:
results_computed2

[   calitp_itp_id                                  organization_name  route_id  \
 0            182  Los Angeles County Metropolitan Transportation...       120   
 
    route_type  
 0           3  ,
    calitp_itp_id     organization_name  route_id  route_type
 0            300  City of Santa Monica        18           1,
    calitp_itp_id                 organization_name  route_id  route_type
 0            282  City and County of San Francisco        67           3]

At this point, you can either write a function to export each individual aggregated pandas df result to be its standalone parquet, or combine it all. 

We will not export and overwrite the file in the GCS bucket right now.

Since our results are just pandas dfs, we could also concatenate them.

In [24]:
pd.concat(results_computed2, axis=0)

,calitp_itp_id,organization_name,route_id,route_type
0,182,Los Angeles County Metropolitan Transportation...,120,3
0,300,City of Santa Monica,18,1
0,282,City and County of San Francisco,67,3


In [25]:
# This is rather pointless for such a small df, but for larger
# ones, we may want to concatenate and export it as a partitioned parquet
dd.multi.concat(results_computed2, axis=0).compute()

,calitp_itp_id,organization_name,route_id,route_type
0,182,Los Angeles County Metropolitan Transportation...,120,3
0,300,City of Santa Monica,18,1
0,282,City and County of San Francisco,67,3


## To Do

* For the same 3 operators, use delayed functions throughout, from importing the parquet, applying a function, and saving the results to a list.
* Your function should group each trip into a category based on its `mean_speed_mph`. 
   * < 10 mph
   * 10-15 mph
   * 15-20 mph
   * 20+ mph
* For each operator, get the count of trips by category and its proportion
* Save the results in a list, compute the results for all the operators at once
* Concatenate the aggregated results for all the operators into one dask df

### Write function

In [45]:
def group_speeds(row):
    if row.mean_speed_mph < 10:
        return "< 10 mph"
    elif 20 < row.mean_speed_mph:
        return "20+ mph"
    elif 10 <= row.mean_speed_mph <=15:
        return "10-15 mph"
    else:
        "16-20 mph"

In [76]:
def aggregate_speeds(df):
    # Apply mph grouping function
    df = df.assign(mph_group = df.apply(lambda x: group_speeds(x), axis=1))
    
    # Grab organization  name
    organization = df.organization_name.iloc[0]
    
    # Find number of rows in df
    df_rows = len(df)
    
    # Summarize
    df = (df
          .groupby(['mph_group'])
          .agg({'trip_id':'nunique'})
          .rename(columns = {'trip_id':'total_trips'})
         )
    
    # Find proportion of trips in each mph group by total trips
    df = df.assign(
        proportion_of_trips = (df.total_trips/df_rows) *100,
        organization_name = organization)
    
    df = df.reset_index()

    return df

In [64]:
# Test with big blue bus
# aggregate_speeds(df)

### Read in the 3 operators

In [65]:
operators

[182, 300, 282]

In [68]:
original_dfs = []
for itp_id in operators:
    og_df = import_data(itp_id)
    original_dfs.append(og_df)

In [71]:
my_df = [compute(i)[0] for i in original_dfs]

In [72]:
my_df = pd.concat(my_df)

In [74]:
my_df.head(1)

,feed_key,trip_key,gtfs_dataset_key,activity_date,trip_id,route_id,route_short_name,shape_id,direction_id,route_type,route_long_name,route_desc,calitp_itp_id,median_time,direction,mean_speed_mph,organization_name
0,1dce186c157f55ed353f9bd8bf6f43b6,fc1b5300b49dc955c5e0b9369798029d,3f3f36b4c41cc6b5df3eb7f5d8ea6e3c,2023-03-15,10120000981334-DEC22,120-13167,120,1200098_DEC22,1,3,Metro Local Line,AVIATION/LAX STA- WHITTWOOD CTR VIA IMPERIAL HWY,182,14:31:38.500000,Westbound,12.930988,Los Angeles County Metropolitan Transportation...


In [80]:
speeds_result = []

for itp_id in operators:
    operator_df = import_data(itp_id)
    aggregated_df = delayed(aggregate_speeds)(operator_df)
    
    speeds_result.append(aggregated_df)

In [81]:
speeds_result

[Delayed('aggregate_speeds-ba81e957-e866-42cc-8166-2c7bc0a0a3b1'),
 Delayed('aggregate_speeds-178aec65-b7fb-4463-a70c-8bf5d92eaf1d'),
 Delayed('aggregate_speeds-aa093e7d-1998-4c6a-9bc3-fc358f30b48c')]

In [82]:
results = [compute(i)[0] for i in speeds_result]

In [85]:
pd.concat(results)

,mph_group,total_trips,proportion_of_trips,organization_name
0,10-15 mph,5806,39.285473,Los Angeles County Metropolitan Transportation...
1,20+ mph,1511,10.223966,Los Angeles County Metropolitan Transportation...
2,< 10 mph,5741,38.845659,Los Angeles County Metropolitan Transportation...
0,10-15 mph,497,40.671031,City of Santa Monica
1,20+ mph,8,0.654664,City of Santa Monica
2,< 10 mph,666,54.500818,City of Santa Monica
0,10-15 mph,1451,16.177946,City and County of San Francisco
1,20+ mph,67,0.747018,City and County of San Francisco
2,< 10 mph,7307,81.469506,City and County of San Francisco
